# SentinelAI: Tier 1 — Unsupervised Anomaly Detection (Isolation Forest)

**Stage**: 06 — Model Training & Threat Classification  
**Architecture Tier**: Tier 1 (Unsupervised Zero-Day Threat Hunting)  
**Input Data**: [`ml/data/processed/sentinel_1M_transformed.parquet`](../data/processed/sentinel_1M_transformed.parquet) ($1,000,000$ flows)  
**Splits Manifest**: [`ml/data/processed/benchmark_splits_manifest.json`](../data/processed/benchmark_splits_manifest.json)  
**Fitted Scaler**: [`ml/saved_models/feature_scaler.joblib`](../saved_models/feature_scaler.joblib)  
**Model Produced**: `ml/saved_models/isolation_forest_model.joblib`  

---

## 🎯 Objectives of Tier 1 Modeling

In enterprise network security, novel zero-day exploits (e.g., Log4Shell, Spring4Shell, unknown buffer overflows) have **zero labeled attack examples** in training databases.

Tier 1 acts as SentinelAI's **Zero-Day Outlier Sensor**:
1. **Baseline Learning**: Train strictly on $290,592$ benign network flows without any attack labels to learn the multi-dimensional manifold of normal enterprise operations.
2. **Outlier Scoring**: Assign continuous anomaly scores based on tree isolation path length.
3. **Operational Threshold Calibration**: Calibrate decision boundaries on validation benign traffic to enforce strict enterprise False Positive Rates ($FPR \le 5\%$ and $FPR \le 1\%$).
4. **Threat Family Profiling**: Empirically evaluate which attack families an unsupervised model detects naturally vs. which stealthy attacks require supervised Tier 2/3 classification.
5. **Zero-Shot Domain Transfer**: Evaluate cross-dataset generalization on the UNSW-NB15 benchmark partition.

In [1]:
import os
import sys
import json
import time
import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# Add repository root to path
sys.path.append(os.path.abspath('..'))
from anomaly_detection.isolation_forest import SentinelAnomalyDetector

np.random.seed(42)
print("Environment and SentinelAnomalyDetector loaded successfully.")

Environment and SentinelAnomalyDetector loaded successfully.


## 1. Load Data Partitions & Fitted Scaler Pipeline

We load `sentinel_1M_transformed.parquet`, `benchmark_splits_manifest.json`, and our leak-free `feature_scaler.joblib`.

In [2]:
data_path = "../data/processed/sentinel_1M_transformed.parquet"
manifest_path = "../data/processed/benchmark_splits_manifest.json"
scaler_path = "../saved_models/feature_scaler.joblib"

t0 = time.time()
df = pq.read_table(data_path).to_pandas()
with open(manifest_path, 'r') as f:
    manifest = json.load(f)
scaler = joblib.load(scaler_path)

feature_cols = manifest['feature_columns']
print(f"Loaded dataset: {len(df):,} flows in {time.time()-t0:.2f}s.")
print(f"Active features ({len(feature_cols)}): {feature_cols}")

Loaded dataset: 1,000,000 flows in 0.85s.
Active features (18): ['dst_port', 'flow_duration_s', 'flow_iat_mean_s', 'fwd_iat_mean_s', 'bwd_iat_mean_s', 'fwd_packets', 'bwd_packets', 'fwd_bytes', 'bwd_bytes', 'fwd_pkt_len_mean', 'bwd_pkt_len_mean', 'flow_bytes_per_sec', 'flow_pkts_per_sec', 'fwd_tcp_win_ratio', 'bwd_tcp_win_ratio', 'traffic_symmetry_ratio', 'pkt_len_ratio', 'bytes_per_packet']


C:\Users\himma\AppData\Roaming\Python\Python312\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 2. Extract Scaled Partitions for Benchmark A & B

In [3]:
# Benchmark A indices
train_idx = manifest['benchmark_A_in_distribution']['train_indices']
val_idx = manifest['benchmark_A_in_distribution']['val_indices']
test_idx = manifest['benchmark_A_in_distribution']['test_indices']

# Filter Train partition to strictly Benign / Normal flows for baseline modeling
train_normal_idx = [i for i in train_idx if df.loc[i, 'attack_family'] == 'Normal']
val_normal_idx = [i for i in val_idx if df.loc[i, 'attack_family'] == 'Normal']

print(f"Training baseline normal flows: {len(train_normal_idx):,} flows")
print(f"Validation normal flows:       {len(val_normal_idx):,} flows")
print(f"Test set total flows:           {len(test_idx):,} flows")

# Scale feature matrices
X_train_normal = scaler.transform(df.loc[train_normal_idx, feature_cols].values)
X_val_normal = scaler.transform(df.loc[val_normal_idx, feature_cols].values)
X_test = scaler.transform(df.loc[test_idx, feature_cols].values)

y_test = df.loc[test_idx, 'label'].values
families_test = df.loc[test_idx, 'attack_family'].values

# Benchmark B (Zero-Shot Cross-Dataset) test partition
cross_test_idx = manifest['benchmark_B_cross_dataset']['test_indices']
X_cross_test = scaler.transform(df.loc[cross_test_idx, feature_cols].values)
y_cross_test = df.loc[cross_test_idx, 'label'].values
families_cross_test = df.loc[cross_test_idx, 'attack_family'].values

print(f"Benchmark B (UNSW-NB15) test flows: {len(cross_test_idx):,} flows")

Training baseline normal flows: 290,592 flows
Validation normal flows:       62,270 flows
Test set total flows:           150,000 flows


C:\Users\himma\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(
C:\Users\himma\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(
C:\Users\himma\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(


Benchmark B (UNSW-NB15) test flows: 92,844 flows


C:\Users\himma\AppData\Roaming\Python\Python312\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(


## 3. Train Baseline Isolation Forest Model

We train `SentinelAnomalyDetector` with $150$ isolation trees and subsample size of $2,048$ samples per tree on the normal traffic manifold.

In [4]:
print(f"Fitting SentinelAnomalyDetector on {len(X_train_normal):,} baseline normal flows...")
t0 = time.time()
detector = SentinelAnomalyDetector(
    n_estimators=150,
    max_samples=2048,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)
detector.fit(X_train_normal, feature_names=feature_cols)
fit_time = time.time() - t0
print(f"Fitting complete in {fit_time:.2f} seconds!")

Fitting SentinelAnomalyDetector on 290,592 baseline normal flows...


Fitting complete in 2.54 seconds!


## 4. Calibrate Operational Decision Thresholds

In security operations centers (SOC), alert fatigue is a primary operational failure mode. A detector that flags $30\%$ false alarms is unusable.

We calibrate two operational thresholds on held-out **validation benign traffic (`X_val_normal`)**:
1. **Balanced Security Threshold ($5\%$ Target FPR)**
2. **Strict Enterprise Threshold ($1\%$ Target FPR)**

In [5]:
thresh_5pct = detector.calibrate_threshold(X_val_normal, target_fpr=0.05)
print(f"Calibrated 5% Target FPR Threshold:  {thresh_5pct:.4f}")

# Calculate 1% target FPR threshold
normal_scores_val = detector.score_samples(X_val_normal)
thresh_1pct = float(np.percentile(normal_scores_val, 99.0))
print(f"Calibrated 1% Target FPR Threshold:  {thresh_1pct:.4f}")

Calibrated 5% Target FPR Threshold:  0.0432


Calibrated 1% Target FPR Threshold:  0.1198


## 5. Evaluate Benchmark A: In-Distribution Test Set ($150,000$ Flows)

We evaluate global discriminative power via **ROC-AUC** and **PR-AUC**, and measure detection recall across each of the 8 canonical threat families.

In [6]:
scores_test = detector.score_samples(X_test)
roc_auc = roc_auc_score(y_test, scores_test)
pr_auc = average_precision_score(y_test, scores_test)

print(f"--- Benchmark A (In-Distribution) Global Discrimination ---")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"  PR-AUC:  {pr_auc:.4f}")

# Evaluate at 5% calibrated threshold
preds_test_5 = (scores_test >= thresh_5pct).astype(int)
normal_fpr_5 = np.mean(preds_test_5[families_test == 'Normal'])
attack_recall_5 = np.mean(preds_test_5[y_test == 1])

# Evaluate at 1% calibrated threshold
preds_test_1 = (scores_test >= thresh_1pct).astype(int)
normal_fpr_1 = np.mean(preds_test_1[families_test == 'Normal'])
attack_recall_1 = np.mean(preds_test_1[y_test == 1])

print(f"\n--- Operational Threshold Performance ---")
print(f"  [5% Setting] Empirical FPR: {normal_fpr_5*100:.2f}% | Overall Attack Recall: {attack_recall_5*100:.2f}%")
print(f"  [1% Setting] Empirical FPR: {normal_fpr_1*100:.2f}% | Overall Attack Recall: {attack_recall_1*100:.2f}%")

--- Benchmark A (In-Distribution) Global Discrimination ---
  ROC-AUC: 0.7091
  PR-AUC:  0.7157

--- Operational Threshold Performance ---
  [5% Setting] Empirical FPR: 4.81% | Overall Attack Recall: 5.01%
  [1% Setting] Empirical FPR: 0.96% | Overall Attack Recall: 0.75%


### Threat Family-by-Family Detection Profile

This analysis reveals the fundamental boundary of pure unsupervised learning: what does an anomaly detector catch vs. miss?

In [7]:
family_rows = []
for fam in np.unique(families_test):
    if fam == 'Normal': continue
    mask = (families_test == fam)
    rec_5 = np.mean(preds_test_5[mask]) * 100
    rec_1 = np.mean(preds_test_1[mask]) * 100
    mean_score = np.mean(scores_test[mask])
    family_rows.append({
        'Canonical Family': fam,
        'Test Flows': int(np.sum(mask)),
        'Mean Anomaly Score': round(mean_score, 4),
        'Recall @ 5% FPR': f"{rec_5:.2f}%",
        'Recall @ 1% FPR': f"{rec_1:.2f}%"
    })

family_df = pd.DataFrame(family_rows).sort_values(by='Mean Anomaly Score', ascending=False)
print("--- Unsupervised Anomaly Detection by Threat Family ---")
print(family_df.to_string(index=False))

--- Unsupervised Anomaly Detection by Threat Family ---


  Canonical Family  Test Flows  Mean Anomaly Score Recall @ 5% FPR Recall @ 1% FPR
Web & Exploitation        2953              0.0326          26.55%           6.40%
   Other / Fuzzing        5630              0.0118           9.79%           1.67%
    Reconnaissance       24717              0.0046           0.71%           0.17%
              DDoS        7500             -0.0152           5.59%           0.23%
       Brute Force        7500             -0.0416           0.00%           0.00%
               DoS        7500             -0.0432          17.95%           1.83%
            Botnet        7500             -0.0606           0.56%           0.00%
      Infiltration       24430             -0.0814           4.40%           0.73%


## 6. Evaluate Benchmark B: Zero-Shot Cross-Dataset ($92,844$ Flows on UNSW-NB15)

We test how well the anomaly detector trained on Canadian/AWS normal traffic transfers to Australian enterprise normal and attack traffic.

In [8]:
scores_cross = detector.score_samples(X_cross_test)
cross_roc_auc = roc_auc_score(y_cross_test, scores_cross)
cross_pr_auc = average_precision_score(y_cross_test, scores_cross)

degradation = (1.0 - (cross_roc_auc / roc_auc)) * 100

print("--- Benchmark B (Zero-Shot Cross-Dataset) Results ---")
print(f"  Zero-Shot ROC-AUC:    {cross_roc_auc:.4f}")
print(f"  Zero-Shot PR-AUC:     {cross_pr_auc:.4f}")
print(f"  ROC-AUC Degradation:  {degradation:.2f}%")

--- Benchmark B (Zero-Shot Cross-Dataset) Results ---
  Zero-Shot ROC-AUC:    0.6986
  Zero-Shot PR-AUC:     0.7444
  ROC-AUC Degradation:  1.48%


## 7. Serialize Trained Detector & Export Artifacts

In [9]:
os.makedirs("../saved_models", exist_ok=True)
model_save_path = "../saved_models/isolation_forest_model.joblib"
detector.save(model_save_path)

file_size_mb = os.path.getsize(model_save_path) / (1024**2)
print(f"Saved trained SentinelAnomalyDetector to {model_save_path} ({file_size_mb:.2f} MB).")
print("Tier 1 Unsupervised Anomaly Detection pipeline complete!")

Saved trained SentinelAnomalyDetector to ../saved_models/isolation_forest_model.joblib (7.19 MB).
Tier 1 Unsupervised Anomaly Detection pipeline complete!
